# 03 · Native hybrid search

Fuse keyword and vector search inside OpenSearch, with one query and no separate entity service.

| | |
| --- | --- |
| **Needs** | `00_setup.ipynb` in the same folder. The first code cell runs it for you. |
| **LLM** | Optional. With `RUN_LLM = False` (the default in `00_setup.ipynb`) you still see everything retrieval does. |
| **Previous / next** | [`02_bm25_ner.ipynb`](02_bm25_ner.ipynb) / [`04_ner_grounding_rerank.ipynb`](04_ner_grounding_rerank.ipynb) |

*The code mirrors the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval) in `instaclustr/labs`. Lines marked "Mirrors" name the file in the journey that each piece comes from. This notebook only searches: it never writes to the cluster.*

### Run the setup

The next cell runs `00_setup.ipynb`. It connects to the cluster, loads the two small models, and defines the helper functions used below. You should see `[PASS]` lines. If you see `[FAIL]`, fix that before going on, using the notes in `00_setup.ipynb`.

In [ ]:
%run ./00_setup.ipynb

## Hybrid RAG (native)

**The idea.** Keep both signals, but fuse them **inside OpenSearch** with one query. A `hybrid` query runs the BM25 `match` and the vector search in parallel. A **search pipeline** then normalizes the two score ranges (min-max) and combines them with weights: **0.3 for BM25, 0.7 for vector**. One index, one request, and no separate NER service.

The pipeline already exists on the cluster. This notebook only **reads** it. (The journey's own `ingest.py` creates it, but this account cannot write.)

*Mirrors:* `scripts/hybrid-rag-native/src/query.py` and `hybrid_search()` in `support/common/opensearch_client.py`.

In [ ]:
pipeline = client.transport.perform_request("GET", "/_search/pipeline/hybrid-search-pipeline")
proc = pipeline["hybrid-search-pipeline"]["phase_results_processors"][0]["normalization-processor"]
print("normalization:", proc["normalization"]["technique"])
print("combination:  ", proc["combination"]["technique"], proc["combination"]["parameters"])

In [ ]:
def hybrid_search(question, k=5):
    body = {"size": k, "_source": RECIPE_FIELDS,
            "query": {"hybrid": {"queries": [
                {"match": {"text": question}},
                {"knn": {"embedding": {"vector": embed(question), "k": k}}},
            ]}}}
    return normalize_hits(search(IDX["hybrid"], body, search_pipeline="hybrid-search-pipeline"))


def ask_hybrid(question, top_k=5):
    hits = hybrid_search(question, top_k)
    prompt = f"Question:\n{question}\n\nContext:\n{build_context(hits) or 'No context available.'}"
    return answer(prompt, VECTOR_SYSTEM), hits


for question, domain in [(Q_GRATEFUL, "thegratefulgirlcooks"), (Q_WOMENS, "womensweeklyfood")]:
    reply, hits = ask_hybrid(question)
    print("\nQ:", question)
    show_hits(hits)
    print(f"  -> the named recipe ({domain}) was retrieved: {has_source(hits, domain)}")
    print("  ANSWER:", reply)

**Expected:** the first cell prints a `min_max` normalization and an `arithmetic_mean` combination with weights `[0.3, 0.7]`. In the second, both Fish and Chips recipes are retrieved for both questions, with no NER service involved. Note that the fused score ranks one of them first for *both* questions: fusion finds the pair, but it does not read the question's website name any better than BM25 does. (Scores are normalized to the 0 to 1 range, so they are not comparable with notebook 01's.)

**The limit to notice.** The combined score is still a *score*. It ranks recipes, but it cannot **guarantee** an exclusion. For a request like "no sulfites", a recipe that says "contains sulfites" can still score high and slip in. Notebook 04 fixes that.

### Checkpoint

You should be able to say what a search pipeline does (it normalizes the two score ranges and combines them 0.3 / 0.7), and why a combined score still cannot guarantee that a recipe is excluded.

Next: open [`04_ner_grounding_rerank.ipynb`](04_ner_grounding_rerank.ipynb).